# 04 — Food Recommender: Training & Evaluation

Trains the model behind `src/diet/recommender.py` (see `scripts/train_food_recommender.py`
for the full methodology) and exports the thesis evidence: a four-way
comparison — hand-tuned linear formula vs. a fitted `LinearRegression` vs.
`RandomForestRegressor` vs. `GradientBoostingRegressor` — evaluated by
`GroupKFold` held out **by patient archetype**, plus a feature-importance
chart for the winning model.

There is no real preference/interaction data in this repo — labels come from
a synthetic patient-archetype grid built on the app's own cited reference
data (`data/reference/*.yaml`), scored by a richer suitability function than
the linear formula it augments (saturation caps, a heme-iron interaction, a
continuous sodium penalty — see `compute_label()` in the training script for
citations). Requires `pip install matplotlib pandas` — analysis-only
dependencies, not part of the runtime app, same as `03_results_figures.ipynb`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
FIGURES = PROJECT_ROOT / "thesis" / "figures"
RESULTS = PROJECT_ROOT / "thesis" / "results"
FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

## Build the synthetic training frame and run the four-way comparison

Reuses `scripts/train_food_recommender.py` directly — this notebook doesn't
reimplement the labeling/evaluation logic, only visualizes its output. Takes
roughly a minute or two (~40k rows × 5-fold `GroupKFold` × four estimators).

In [ ]:
import numpy as np

import scripts.train_food_recommender as train

rng = np.random.default_rng(train.RANDOM_STATE)
X, y, groups, baseline = train.build_training_frame(rng)
print(f"{X.shape[0]} rows, {X.shape[1]} features, {len(np.unique(groups))} archetypes")

summary = train.evaluate(X, y, groups, baseline)
summary

## Metrics table → thesis/results

The key ablation evidence: does the ensemble beat both the untrained
hand-tuned formula *and* a plain fitted linear model on patient archetypes
it never trained on?

In [ ]:
import pandas as pd

rows = []
for model_name, metrics in summary.items():
    rows.append({
        "model": model_name,
        "spearman_mean": metrics["spearman"]["mean"],
        "spearman_std": metrics["spearman"]["std"],
        "r2_mean": metrics["r2"]["mean"],
        "mae_mean": metrics["mae"]["mean"],
    })
table = pd.DataFrame(rows).sort_values("spearman_mean", ascending=False)
table.to_csv(RESULTS / "recommender_model_comparison.csv", index=False)
table

## Feature importance → thesis/figures

Loads the shipped model (`data/models/food_recommender.joblib`, produced by
running the training script from the project's own `.venv`) rather than
refitting here, so the chart matches what's actually deployed.

In [ ]:
import matplotlib.pyplot as plt

from src.diet import recommender

model = recommender.load_model()
assert model is not None, "run scripts/train_food_recommender.py first"

importances = pd.Series(model.feature_importances_, index=recommender.FEATURE_NAMES)
top = importances.sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(7, 5))
top.plot.barh(ax=ax)
ax.set_xlabel("feature importance")
ax.set_title("Food recommender — top 15 features")
fig.tight_layout()
fig.savefig(FIGURES / "recommender_feature_importance.png", dpi=200, bbox_inches="tight")

## Predicted vs. baseline → thesis/figures

Where the trained model and the hand-tuned formula agree vs. diverge, over
the full synthetic frame (not just one held-out fold, purely for the
visual).

In [ ]:
predicted = model.predict(X)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(baseline, predicted, s=4, alpha=0.15)
lims = [min(baseline.min(), predicted.min()), max(baseline.max(), predicted.max())]
ax.plot(lims, lims, color="black", linewidth=1, linestyle="--", label="y = x")
ax.set_xlabel("hand-tuned formula score (baseline)")
ax.set_ylabel("trained model score")
ax.set_title("Trained model vs. hand-tuned baseline")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "recommender_predicted_vs_baseline.png", dpi=200, bbox_inches="tight")